# Paper-Ready Metric Plots

Plots PSNR and SSIM vs transmission budget from `metrics/summary.csv`.
Error bars show **95% CI of the mean**: `mean ± 1.96 × (σ/√n)` — how precisely
the mean is estimated across views, not the spread of individual views.

**Run non-interactively from `dlapisgs-utility/`:**
```bash
jupyter nbconvert --to notebook --execute --inplace \
    --ExecutePreprocessor.timeout=120 \
    plotting/paper_plot_metrics.ipynb
```

## Parameters
Edit this cell to point at different data or change visual settings.
When running via `nbconvert`, override with `papermill` or by editing here.

In [1]:
# ── I/O ──────────────────────────────────────────────────────────────────────
# Paths are relative to the dlapisgs-utility/ project root.
# The working-directory cell below sets cwd to that root automatically.
SUMMARY_CSV = "output/0513_setup2_progressive/metrics/summary.csv"
OUT_DIR     = "plotting/paper"

# Column in the CSV to group lines by.
# Use 'weight_mode' for setup2, 'scheme' for scheme comparisons, etc.
GROUP_BY = "weight_mode"

# Keys to drop before plotting (implementation mistakes, deprecated modes, …)
EXCLUDE_KEYS = ["det_gamma_over_d2"]

# ── Per-key display config ───────────────────────────────────────────────────
# Keys not listed here get auto-assigned label/marker/color.
KEY_CONFIG = {
    # weight_mode keys
    "volume":         {"label": "Volume (view-indep.)",   "marker": "^", "color": "#2ca02c"},
    "volume_over_d2": {"label": r"Vol/d$^2$ (view-dep.)", "marker": "D", "color": "#d62728"},
    "screen_area":    {"label": "Screen Area",             "marker": "o", "color": "#9467bd"},
    # scheme keys
    "vd_lod":         {"label": "VD+LOD (baseline)",       "marker": "s", "color": "#1f77b4"},
    "vd_lod_w":       {"label": "VD+LOD+W",                "marker": "^", "color": "#ff7f0e"},
    "vd_lod_c":       {"label": "VD+LOD+C",                "marker": "D", "color": "#2ca02c"},
    "vd_lod_w_c":     {"label": "VD+LOD+W+C (proposed)",   "marker": "o", "color": "#d62728"},
}

# Preferred key order for legend / line draw order (best → worst, or logical)
KEY_ORDER = {
    "weight_mode": ["screen_area", "volume_over_d2", "volume"],
    "scheme":      ["vd_lod", "vd_lod_w", "vd_lod_c", "vd_lod_w_c"],
}

# ── Figure style ─────────────────────────────────────────────────────────────
FIGSIZE    = (6.4, 4.8)
FONT_SIZE  = 18
LEGEND_FS  = 13
TICK_FS    = 14
LINE_W     = 1.8
MARKER_SZ  = 7
ERR_LW     = 1.5
ERR_CAP    = 4
ERR_CAPTHK = 1.5
DPI        = 300
GRID_ALPHA = 0.25

## Working Directory

In [2]:
import os
from pathlib import Path

# Ensure we run relative to dlapisgs-utility/ regardless of where
# nbconvert was invoked from.
_cwd = Path(os.getcwd())
_root = None
for candidate in [_cwd, _cwd.parent]:
    if (candidate / "utility_calculation.py").exists():
        _root = candidate
        break

if _root is None:
    # nbconvert temp-script sets __file__; notebook lives in plotting/
    try:
        _root = Path(__file__).resolve().parent.parent
    except NameError:
        _root = _cwd

os.chdir(_root)
print(f"Working directory: {Path.cwd()}")

Working directory: /mnt/data1/samk/gs-quic/cs5262_tile_quic/dlapisgs-utility


## Imports & Style

In [3]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt

matplotlib.use("Agg")

# Times New Roman + LaTeX rendering — matches plotter.ipynb
plt.rc("text", usetex=True)
plt.rc("font", family="Times New Roman", serif="Times", size=FONT_SIZE)

# Seaborn base theme
sns.set_theme(style="whitegrid", context="paper", font="Times New Roman")

print(f"seaborn {sns.__version__}, matplotlib {matplotlib.__version__}")

seaborn 0.13.2, matplotlib 3.10.6


## Load & Aggregate

In [4]:
summary_csv = Path(SUMMARY_CSV)
if not summary_csv.exists():
    raise FileNotFoundError(f"Summary CSV not found: {summary_csv}  (cwd={Path.cwd()})")

df = pd.read_csv(summary_csv)
print(f"Loaded {len(df)} rows")
print(f"Columns : {list(df.columns)}")
print(f"Group-by: {GROUP_BY}  →  {sorted(df[GROUP_BY].unique())}")
print(f"Budgets : {sorted(df['budget_mb'].unique())}")

df = df[~df[GROUP_BY].isin(EXCLUDE_KEYS)].copy()
print(f"After exclusions: {sorted(df[GROUP_BY].unique())}")

Loaded 1400 rows
Columns : ['budget_mb', 'scheme', 'camera_index', 'psnr', 'ssim', 'used_bytes', 'selected_gaussians', 'ply_bytes', 'w_norm', 'c_norm', 'packing_mode', 'weight_mode']
Group-by: weight_mode  →  ['det_gamma_over_d2', 'screen_area', 'volume', 'volume_over_d2']
Budgets : [np.float64(20.0), np.float64(60.0), np.float64(100.0), np.float64(200.0), np.float64(500.0), np.float64(700.0), np.float64(1000.0)]
After exclusions: ['screen_area', 'volume', 'volume_over_d2']


In [5]:
def aggregate_ci95(df: pd.DataFrame, group_by: str) -> pd.DataFrame:
    """Per-(group, budget) mean and 95% CI half-width.

    95% CI = mean ± 1.96 × (σ / √n)
    Parametric normal approximation; valid by CLT for n ≥ 30.
    """
    records = []
    for key, gdf in df.groupby(group_by):
        for budget, bdf in gdf.groupby("budget_mb"):
            for metric in ("psnr", "ssim"):
                vals = bdf[metric].dropna().values
                n    = len(vals)
                mean = vals.mean()
                ci   = 1.96 * vals.std(ddof=1) / np.sqrt(n)
                records.append({
                    group_by:    key,
                    "budget_mb": float(budget),
                    "metric":    metric,
                    "mean":      mean,
                    "ci95":      ci,
                    "n":         n,
                })
    return pd.DataFrame(records)

agg = aggregate_ci95(df, GROUP_BY)
print(agg.head(12).to_string(index=False))

weight_mode  budget_mb metric      mean     ci95  n
screen_area       20.0   psnr 12.067616 1.051026 50
screen_area       20.0   ssim  0.303152 0.063834 50
screen_area       60.0   psnr 13.869164 1.451623 50
screen_area       60.0   ssim  0.399806 0.078975 50
screen_area      100.0   psnr 15.228505 1.752886 50
screen_area      100.0   ssim  0.461441 0.086532 50
screen_area      200.0   psnr 18.281535 2.389487 50
screen_area      200.0   ssim  0.568526 0.101364 50
screen_area      500.0   psnr 26.493134 4.534499 50
screen_area      500.0   ssim  0.641896 0.115521 50
screen_area      700.0   psnr 34.713965 7.292917 50
screen_area      700.0   ssim  0.643762 0.115965 50


## Plot

In [6]:
def resolve_order(keys):
    preferred = KEY_ORDER.get(GROUP_BY, [])
    ordered   = [k for k in preferred if k in keys]
    extras    = [k for k in sorted(keys) if k not in ordered]
    return ordered + extras


def plot_metric(agg, metric, ylabel, out_stem, out_dir):
    sub    = agg[agg["metric"] == metric]
    keys   = resolve_order(sub[GROUP_BY].unique().tolist())

    fallback_colors  = sns.color_palette("tab10", max(len(keys), 1))
    fallback_markers = ["s", "^", "D", "o", "v", "P", "X"]

    fig, ax = plt.subplots(figsize=FIGSIZE)

    for i, key in enumerate(keys):
        cfg    = KEY_CONFIG.get(key, {})
        label  = cfg.get("label",  key)
        marker = cfg.get("marker", fallback_markers[i % len(fallback_markers)])
        color  = cfg.get("color",  fallback_colors[i])

        kdf = sub[sub[GROUP_BY] == key].sort_values("budget_mb")
        ax.errorbar(
            kdf["budget_mb"].values,
            kdf["mean"].values,
            yerr=kdf["ci95"].values,
            marker=marker, color=color,
            linewidth=LINE_W, markersize=MARKER_SZ,
            capsize=ERR_CAP, elinewidth=ERR_LW, capthick=ERR_CAPTHK,
            label=label,
        )

    ax.set_xlabel(r"Budget (MiB)", fontsize=FONT_SIZE)
    ax.set_ylabel(ylabel,          fontsize=FONT_SIZE)
    ax.tick_params(labelsize=TICK_FS)
    ax.legend(fontsize=LEGEND_FS, framealpha=0.9)
    ax.grid(alpha=GRID_ALPHA)
    sns.despine(ax=ax, left=False, bottom=False)
    fig.tight_layout()

    out_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_dir / f"{out_stem}.png", dpi=DPI, bbox_inches="tight")
    fig.savefig(out_dir / f"{out_stem}.eps", format="eps", bbox_inches="tight")
    print(f"Wrote {out_dir}/{out_stem}.{{png,eps}}")
    plt.close(fig)

In [7]:
out_dir = Path(OUT_DIR)

plot_metric(agg, "psnr", r"PSNR (dB)", "psnr_vs_budget", out_dir)
plot_metric(agg, "ssim", r"SSIM",       "ssim_vs_budget", out_dir)

print("Done.")

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/psnr_vs_budget.{png,eps}


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/ssim_vs_budget.{png,eps}
Done.
